# ActionFormer + BiMamba-Transformer on THUMOS14 (RunPod)

**Works with any RunPod PyTorch template.** Uses system Python directly — no venv.

## Step 0: Detect Environment

In [ ]:
import torch, sys, os, subprocess

print(f"Python:       {sys.executable}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA (torch): {torch.version.cuda}")
print(f"CUDA avail:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM:         {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Detect system CUDA
r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
if r.returncode == 0:
    cuda_line = [l for l in r.stdout.split('\n') if 'release' in l]
    print(f"System CUDA:  {cuda_line[0].strip() if cuda_line else 'unknown'}")

# Extract major.minor CUDA version for later use
TORCH_CUDA = torch.version.cuda  # e.g. "12.4"
TORCH_VER = torch.__version__.split('+')[0]  # e.g. "2.4.1"
print(f"\nWill install mamba-ssm compatible with torch {TORCH_VER} + CUDA {TORCH_CUDA}")

## Step 1: Install Mamba (version-matched to YOUR PyTorch)

In [ ]:
import torch, subprocess, sys

torch_major_minor = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
print(f"Detected PyTorch {torch_major_minor[0]}.{torch_major_minor[1]}")

# Pick mamba version that matches this PyTorch
if torch_major_minor >= (2, 6):
    MAMBA_VER = "mamba-ssm>=2.3.0"
elif torch_major_minor >= (2, 4):
    MAMBA_VER = "mamba-ssm==2.2.2"
elif torch_major_minor >= (2, 1):
    MAMBA_VER = "mamba-ssm==1.2.2"
else:
    MAMBA_VER = "mamba-ssm==1.1.4"

print(f"Selected: {MAMBA_VER}")

In [ ]:
%%bash -e
# Use system pip directly — no venv, no build isolation
# --no-build-isolation ensures it uses the EXISTING system torch for compilation
# --no-deps prevents it from pulling a different torch version

echo "=== Step 1a: Install build deps ==="
pip install packaging ninja einops --quiet 2>&1 | tail -3

echo "=== Step 1b: Install causal-conv1d ==="
pip install causal-conv1d --no-build-isolation 2>&1 | tail -5

echo "=== Step 1c: Detect torch version and install matching mamba ==="
TORCH_VER=$(python3 -c "import torch; v=torch.__version__.split('+')[0].split('.'); print(f'{v[0]}.{v[1]}')")
echo "PyTorch major.minor: $TORCH_VER"

case $TORCH_VER in
    2.4|2.5)  MAMBA="mamba-ssm==2.2.2" ;;
    2.6|2.7)  MAMBA="mamba-ssm>=2.3.0" ;;
    2.1|2.2|2.3) MAMBA="mamba-ssm==1.2.2" ;;
    *)        MAMBA="mamba-ssm==2.2.2"
              echo "Unknown torch $TORCH_VER, trying mamba 2.2.2..." ;;
esac

echo "Installing: $MAMBA"
pip install $MAMBA --no-build-isolation --no-deps 2>&1 | tail -5

echo "=== Step 1d: Install mamba's runtime deps (WITHOUT touching torch) ==="
pip install einops transformers --quiet 2>&1 | tail -3

echo "=== Done ==="

In [ ]:
# Verify mamba works
import torch
from mamba_ssm import Mamba

print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")

m = Mamba(d_model=256, d_state=16, d_conv=4, expand=2).cuda()
x = torch.randn(1, 100, 256).cuda()
y = m(x)
print(f"Mamba:   {x.shape} → {y.shape} ✓")
del m, x, y
torch.cuda.empty_cache()

## Step 2: Clone ActionFormer + Download THUMOS14 Features

In [ ]:
import os, subprocess

WORK = "/workspace/bimamba_tal"
AF = f"{WORK}/actionformer_release"
os.makedirs(WORK, exist_ok=True)

if not os.path.isdir(AF):
    print("Cloning ActionFormer...")
    subprocess.run(["git", "clone", "https://github.com/happyharrycn/actionformer_release.git", AF], check=True)
    print("Done")
else:
    print(f"Already cloned at {AF}")

# Install ActionFormer deps
subprocess.run(["pip", "install", "-q", "pyyaml", "h5py", "joblib", 
                "tensorboard", "pandas", "scipy"])

print("\nActionFormer structure:")
for f in sorted(os.listdir(AF)):
    print(f"  {f}")

In [ ]:
import os, subprocess, glob

AF = "/workspace/bimamba_tal/actionformer_release"
FEAT = f"{AF}/data/thumos/i3d_features"

# Check if features already exist
existing = glob.glob(f"{FEAT}/*.npy") if os.path.isdir(FEAT) else []

if len(existing) > 0:
    print(f"Features already present: {len(existing)} files")
else:
    print("Downloading THUMOS14 I3D features...")
    os.makedirs(os.path.dirname(FEAT), exist_ok=True)
    subprocess.run(["pip", "install", "-q", "gdown"], check=True)
    
    # Download from ActionFormer's Google Drive
    # File ID from: https://github.com/happyharrycn/actionformer_release#data-preparation
    r = subprocess.run([
        "gdown", "1kQMr60i9SATW2fzdjsGCMPORoSR8SLiz", 
        "-O", f"{AF}/data/thumos/thumos_i3d.tar.gz"
    ])
    
    if r.returncode == 0 and os.path.exists(f"{AF}/data/thumos/thumos_i3d.tar.gz"):
        print("Extracting...")
        subprocess.run(["tar", "-xzf", f"{AF}/data/thumos/thumos_i3d.tar.gz", 
                       "-C", f"{AF}/data/thumos/"], check=True)
        print("Done")
    else:
        print("\n" + "="*60)
        print("AUTO-DOWNLOAD FAILED (Google Drive rate limits)")
        print("="*60)
        print("Manual steps:")
        print("1. Go to: https://github.com/happyharrycn/actionformer_release")
        print("2. Download THUMOS14 I3D features from the README links")
        print(f"3. Extract into: {AF}/data/thumos/")
        print(f"4. Ensure .npy files are in: {FEAT}/")
        print("5. Re-run this cell to verify")

In [ ]:
# Verify features and annotations
import os, glob, json, numpy as np

AF = "/workspace/bimamba_tal/actionformer_release"

# Find features
npy = glob.glob(f"{AF}/data/thumos/**/*.npy", recursive=True)
print(f"Feature files: {len(npy)}")
if npy:
    s = np.load(npy[0])
    FEAT_DIR = os.path.dirname(npy[0])
    print(f"  Shape: {s.shape} (expect [T, 2048])")
    print(f"  Directory: {FEAT_DIR}")

# Find annotations
anno = glob.glob(f"{AF}/data/thumos/**/*.json", recursive=True)
print(f"\nAnnotation files: {anno}")
if anno:
    with open(anno[0]) as f:
        d = json.load(f)
    db = d.get('database', d)
    ANNO_FILE = anno[0]
    print(f"  Videos: {len(db)}")

assert len(npy) > 0, "NO FEATURES FOUND. Download them first (see cell above)."
assert len(anno) > 0, "NO ANNOTATIONS FOUND."
print("\n✓ Data ready")

## Step 3: Inspect ActionFormer's Backbone Interface

We need to understand the exact API before writing our replacement.

In [ ]:
import os, re

AF = "/workspace/bimamba_tal/actionformer_release"

# Read key source files
with open(f"{AF}/libs/modeling/backbones.py") as f:
    bb_src = f.read()
with open(f"{AF}/libs/modeling/meta_archs.py") as f:
    ma_src = f.read()
with open(f"{AF}/libs/modeling/__init__.py") as f:
    init_src = f.read()

# Show the registration system
print("=== Backbone Registry ===")
for m in re.finditer(r'(def register_backbone.*?\n(?:.*?\n)*?.*?return)', bb_src):
    print(m.group()[:300])

print("\n=== make_backbone ===")
for m in re.finditer(r'(def make_backbone.*?\n(?:.*?\n)*?.*?return.*?\n)', bb_src):
    print(m.group()[:400])

print("\n=== Backbone instantiation in meta_archs ===")
for m in re.finditer(r'.*make_backbone.*', ma_src):
    print(m.group().strip())

print("\n=== How backbone kwargs are built ===")
# Find the dict that's passed to make_backbone
idx = ma_src.find('make_backbone')
if idx > 0:
    chunk = ma_src[idx:idx+800]
    print(chunk)

In [ ]:
# Also check the original ConvTransformerBackbone signature
import re

AF = "/workspace/bimamba_tal/actionformer_release"
with open(f"{AF}/libs/modeling/backbones.py") as f:
    bb_src = f.read()

# Find the __init__ signature
match = re.search(r'class ConvTransformerBackbone.*?def __init__\(self,(.*?)\):', bb_src, re.DOTALL)
if match:
    params = match.group(1)
    print("ConvTransformerBackbone.__init__ params:")
    for line in params.strip().split('\n'):
        print(f"  {line.strip()}")

# Find forward signature and return
match = re.search(r'def forward\(self.*?return.*?\n', bb_src, re.DOTALL)
if match:
    print(f"\nforward returns: see source")

## Step 4: Write BiMamba-Transformer Backbone

This cell writes the backbone file. It uses `@register_backbone` to hook into ActionFormer's registry.

The backbone accepts `**kwargs` so it silently ignores any extra params ActionFormer passes.

In [ ]:
AF = "/workspace/bimamba_tal/actionformer_release"

code = '''"""BiMamba-Transformer backbone for ActionFormer."""

import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm import Mamba
from .blocks import MaskedConv1D, LayerNorm
from .backbones import register_backbone


class BiMambaLayer(nn.Module):
    """Forward + backward Mamba scanning, fused by summation (ViM-style)."""

    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.mamba_fwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.mamba_bwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x, mask):
        # x: (B, C, T)  mask: (B, 1, T)
        B, C, T = x.shape
        xt = x.permute(0, 2, 1)                          # (B, T, C)
        if mask is not None:
            m = mask.squeeze(1).unsqueeze(-1).float()     # (B, T, 1)
            xt = xt * m

        fwd = self.mamba_fwd(xt)
        bwd = self.mamba_bwd(xt.flip(1)).flip(1)
        out = fwd + bwd

        if mask is not None:
            out = out * m
        return out.permute(0, 2, 1)                      # (B, C, T)


class LocalMHA(nn.Module):
    """Sliding-window multi-head self-attention."""

    def __init__(self, d_model, n_heads, window_size, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head ** -0.5
        self.win = window_size
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        B, C, T = x.shape
        xt = x.permute(0, 2, 1)
        qkv = self.qkv(xt).reshape(B, T, 3, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q * self.scale) @ k.transpose(-2, -1)

        # window mask
        if 0 < self.win < T:
            pos = torch.arange(T, device=x.device)
            wmask = ((pos.unsqueeze(0) - pos.unsqueeze(1)).abs() <= self.win).float()
            wmask = wmask.unsqueeze(0).unsqueeze(0)
            if mask is not None:
                wmask = wmask * mask.unsqueeze(2).float()
            attn = attn.masked_fill(wmask == 0, -1e9)
        elif mask is not None:
            attn = attn.masked_fill(mask.unsqueeze(2).float() == 0, -1e9)

        attn = self.drop(F.softmax(attn, dim=-1))
        attn = torch.nan_to_num(attn)
        out = (attn @ v).permute(0, 2, 1, 3).reshape(B, T, C)
        out = self.proj(out)
        if mask is not None:
            out = out * mask.squeeze(1).unsqueeze(-1).float()
        return out.permute(0, 2, 1)


class BiMambaTransformerBlock(nn.Module):
    """LN -> BiMamba -> res -> LN -> LocalMHA -> res -> LN -> FFN -> res"""

    def __init__(self, d, n_heads, win, d_state=16, d_conv=4,
                 mamba_expand=2, ffn_expand=4, drop=0.0):
        super().__init__()
        self.ln1 = LayerNorm(d)
        self.bimamba = BiMambaLayer(d, d_state, d_conv, mamba_expand)
        self.dp1 = nn.Dropout(drop)

        self.ln2 = LayerNorm(d)
        self.attn = LocalMHA(d, n_heads, win, drop)
        self.dp2 = nn.Dropout(drop)

        self.ln3 = LayerNorm(d)
        self.ffn = nn.Sequential(
            nn.Conv1d(d, d * ffn_expand, 1), nn.GELU(), nn.Dropout(drop),
            nn.Conv1d(d * ffn_expand, d, 1), nn.Dropout(drop),
        )

    def forward(self, x, mask):
        x = x + self.dp1(self.bimamba(self.ln1(x), mask))
        x = x + self.dp2(self.attn(self.ln2(x), mask))
        x = x + self.ffn(self.ln3(x))
        if mask is not None:
            x = x * mask.float()
        return x, mask


@register_backbone("convBiMambaTransformer")
class ConvBiMambaTransformerBackbone(nn.Module):
    """
    Drop-in replacement for ConvTransformerBackbone.
    Same interface: (x, mask) -> ([feats], [masks])
    Accepts **kwargs to silently ignore extra config params.
    """

    def __init__(self, n_in, n_embd, n_head, n_embd_ks, max_len,
                 arch=(2, 2, 5), mha_win_size=None, scale_factor=2,
                 with_ln=False, attn_pdrop=0.0, proj_pdrop=0.0,
                 path_pdrop=0.0, use_abs_pe=False, use_rel_pe=False,
                 # BiMamba params (with defaults so it works even if not in config)
                 d_state=16, d_conv=4, mamba_expand=2,
                 **kwargs):  # absorb anything else ActionFormer passes
        super().__init__()
        if mha_win_size is None:
            mha_win_size = [-1] * 6
        n_levels, n_blocks, ds_factor = arch

        # Feature embedding (identical to ActionFormer)
        self.embd = nn.ModuleList()
        self.embd_norm = nn.ModuleList()
        for i in range(ds_factor):
            ic = n_in if i == 0 else n_embd
            self.embd.append(MaskedConv1D(
                ic, n_embd, n_embd_ks,
                stride=1, padding=n_embd_ks // 2, bias=(not with_ln)))
            self.embd_norm.append(LayerNorm(n_embd) if with_ln else nn.Identity())

        self.use_abs_pe = use_abs_pe
        if use_abs_pe:
            self.abs_pe = nn.Parameter(torch.zeros(1, n_embd, max_len))
            nn.init.trunc_normal_(self.abs_pe, std=0.02)

        # Multi-scale pyramid
        self.n_levels = n_levels
        self.ds = nn.ModuleList()
        self.ds_norm = nn.ModuleList()
        self.blocks = nn.ModuleList()

        for lvl in range(n_levels):
            # Downsample conv (skip level 0)
            if lvl > 0:
                self.ds.append(MaskedConv1D(
                    n_embd, n_embd, scale_factor + 1,
                    stride=scale_factor,
                    padding=(scale_factor + 1) // 2,
                    bias=(not with_ln)))
                self.ds_norm.append(LayerNorm(n_embd) if with_ln else nn.Identity())
            else:
                self.ds.append(nn.Identity())
                self.ds_norm.append(nn.Identity())

            # Encoder blocks
            w = mha_win_size[lvl] if lvl < len(mha_win_size) else -1
            if w <= 0:
                w = 99999

            lvl_blocks = nn.ModuleList()
            for _ in range(n_blocks):
                lvl_blocks.append(BiMambaTransformerBlock(
                    d=n_embd, n_heads=n_head, win=w,
                    d_state=d_state, d_conv=d_conv,
                    mamba_expand=mamba_expand,
                    drop=proj_pdrop,
                ))
            self.blocks.append(lvl_blocks)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Conv1d, nn.Linear)):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x, mask):
        # Embedding
        for embd, norm in zip(self.embd, self.embd_norm):
            x, mask = embd(x, mask)
            x = F.relu(norm(x))

        if self.use_abs_pe:
            x = x + self.abs_pe[:, :, :x.shape[-1]]

        # Pyramid levels
        out_feats, out_masks = [], []
        for lvl in range(self.n_levels):
            if lvl > 0:
                x, mask = self.ds[lvl](x, mask)
                x = F.relu(self.ds_norm[lvl](x))
            for blk in self.blocks[lvl]:
                x, mask = blk(x, mask)
            out_feats.append(x)
            out_masks.append(mask)

        return out_feats, out_masks
'''

path = f"{AF}/libs/modeling/bimamba_backbone.py"
with open(path, 'w') as f:
    f.write(code)
print(f"Written: {path}")

# Add import to __init__.py
init_path = f"{AF}/libs/modeling/__init__.py"
with open(init_path, 'r') as f:
    init = f.read()
if 'bimamba_backbone' not in init:
    with open(init_path, 'a') as f:
        f.write("\nfrom .bimamba_backbone import ConvBiMambaTransformerBackbone\n")
    print(f"Patched: {init_path}")
else:
    print(f"Already patched: {init_path}")

## Step 5: Smoke-test the backbone

In [ ]:
import sys, torch
sys.path.insert(0, "/workspace/bimamba_tal/actionformer_release")

from libs.modeling.bimamba_backbone import ConvBiMambaTransformerBackbone

backbone = ConvBiMambaTransformerBackbone(
    n_in=2048, n_embd=512, n_head=4, n_embd_ks=3, max_len=2304,
    arch=(6, 2, 5), mha_win_size=[19]*6, scale_factor=2,
    with_ln=True, proj_pdrop=0.0, d_state=16, d_conv=4, mamba_expand=2,
).cuda()

n_params = sum(p.numel() for p in backbone.parameters()) / 1e6
print(f"Parameters: {n_params:.1f}M")

x = torch.randn(2, 2048, 2304).cuda()
mask = torch.ones(2, 1, 2304, dtype=torch.bool).cuda()

with torch.no_grad():
    feats, masks = backbone(x, mask)

print("Pyramid output:")
for i, (f, m) in enumerate(zip(feats, masks)):
    print(f"  Level {i}: {tuple(f.shape)}, mask {tuple(m.shape)}")

del backbone, x, mask, feats, masks
torch.cuda.empty_cache()
print("\n✓ Backbone works")

## Step 6: Create Config

In [ ]:
import yaml, copy, os, glob

AF = "/workspace/bimamba_tal/actionformer_release"

# Load ActionFormer's original THUMOS config
with open(f"{AF}/configs/thumos_i3d.yaml") as f:
    cfg = yaml.safe_load(f)

# Fix data paths to match actual locations on this pod
npy_files = glob.glob(f"{AF}/data/thumos/**/*.npy", recursive=True)
anno_files = glob.glob(f"{AF}/data/thumos/**/*.json", recursive=True)
assert npy_files, "No features found!"
assert anno_files, "No annotations found!"

cfg['dataset']['feat_folder'] = os.path.dirname(npy_files[0])
cfg['dataset']['json_file'] = anno_files[0]

# ONLY change: swap backbone type
cfg['model']['backbone_type'] = 'convBiMambaTransformer'

# Save
out_path = f"{AF}/configs/thumos_bimamba.yaml"
with open(out_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"Config saved: {out_path}")
print(f"\nKey settings:")
print(f"  backbone_type: {cfg['model']['backbone_type']}")
print(f"  backbone_arch: {cfg['model']['backbone_arch']}")
print(f"  embd_dim:      {cfg['model']['embd_dim']}")
print(f"  n_head:        {cfg['model']['n_head']}")
print(f"  mha_win_size:  {cfg['model']['n_mha_win_size']}")
print(f"  feat_folder:   {cfg['dataset']['feat_folder']}")
print(f"  json_file:     {cfg['dataset']['json_file']}")
print(f"  batch_size:    {cfg['loader']['batch_size']}")
print(f"  epochs:        {cfg['opt']['epochs']}")
print(f"  lr:            {cfg['opt']['learning_rate']}")

## Step 7: Verify the Full Model Builds

This tests that ActionFormer can construct the complete model (backbone + heads) from our config.

In [ ]:
import sys, yaml, torch
sys.path.insert(0, "/workspace/bimamba_tal/actionformer_release")

AF = "/workspace/bimamba_tal/actionformer_release"

# Import ActionFormer's model builder
from libs.modeling import make_meta_arch

with open(f"{AF}/configs/thumos_bimamba.yaml") as f:
    cfg = yaml.safe_load(f)

# Build the full model
try:
    model = make_meta_arch(cfg['model']['backbone_type'], **cfg['model'])
    print(f"✓ Full model built via make_meta_arch")
except Exception as e1:
    print(f"make_meta_arch failed: {e1}")
    print("Trying alternative build path...")
    try:
        # Some ActionFormer versions use a different API
        from libs.modeling import make_backbone
        backbone = make_backbone('convBiMambaTransformer', **{
            'n_in': cfg['dataset']['input_dim'],
            'n_embd': cfg['model']['embd_dim'],
            'n_head': cfg['model']['n_head'],
            'n_embd_ks': cfg['model']['embd_kernel_size'],
            'max_len': cfg['dataset']['max_seq_len'],
            'arch': cfg['model']['backbone_arch'],
            'mha_win_size': cfg['model']['n_mha_win_size'],
            'scale_factor': cfg['model']['scale_factor'],
            'with_ln': cfg['model']['embd_with_ln'],
        })
        print(f"✓ Backbone built via make_backbone")
    except Exception as e2:
        print(f"make_backbone also failed: {e2}")
        print("\nThis means ActionFormer's model builder doesn't pass our")
        print("backbone_type to make_backbone. Check Step 3 output for the")
        print("exact mechanism and adjust accordingly.")

## Step 8: Train

Expected training time per GPU:
- **L4 (24GB)**: ~3-4 min/epoch, ~2 hours total
- **RTX 3090 (24GB)**: ~2-3 min/epoch, ~1.5 hours total
- **L40 (48GB)**: ~2 min/epoch, ~1 hour total

In [ ]:
import subprocess, os

AF = "/workspace/bimamba_tal/actionformer_release"
CONFIG = f"{AF}/configs/thumos_bimamba.yaml"
OUTPUT = "/workspace/bimamba_tal/ckpt"
os.makedirs(OUTPUT, exist_ok=True)

# Find the train script
train_py = None
for name in ["train.py", "tools/train.py", "scripts/train.py"]:
    path = os.path.join(AF, name)
    if os.path.exists(path):
        train_py = path
        break

assert train_py, f"Cannot find train.py in {AF}"
print(f"Train script: {train_py}")
print(f"Config:       {CONFIG}")
print(f"Output:       {OUTPUT}")
print()

# Run training
proc = subprocess.Popen(
    ["python", train_py, CONFIG, "--output", OUTPUT],
    cwd=AF,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in iter(proc.stdout.readline, ''):
    print(line, end='')

proc.wait()
print(f"\nExit code: {proc.returncode}")

## Step 9: Evaluate

In [ ]:
import subprocess, os, glob

AF = "/workspace/bimamba_tal/actionformer_release"
CONFIG = f"{AF}/configs/thumos_bimamba.yaml"
OUTPUT = "/workspace/bimamba_tal/ckpt"

# Find eval script
eval_py = None
for name in ["eval.py", "tools/eval.py", "scripts/eval.py"]:
    path = os.path.join(AF, name)
    if os.path.exists(path):
        eval_py = path
        break

# Find latest checkpoint
ckpts = sorted(glob.glob(f"{OUTPUT}/*.pth.tar"))
if not ckpts:
    ckpts = sorted(glob.glob(f"{OUTPUT}/**/*.pth.tar", recursive=True))

if ckpts and eval_py:
    ckpt = ckpts[-1]
    print(f"Evaluating: {ckpt}")
    proc = subprocess.Popen(
        ["python", eval_py, CONFIG, ckpt],
        cwd=AF,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in iter(proc.stdout.readline, ''):
        print(line, end='')
    proc.wait()
else:
    print(f"Checkpoints found: {ckpts}")
    print(f"Eval script: {eval_py}")
    print("Train the model first (Step 8).")

## Step 10: Tuning Guide

If your avg mAP is below 65%, try these changes **one at a time**:

| Priority | Parameter | Default | Try | Where |
|----------|-----------|---------|-----|-------|
| 1 | `learning_rate` | 1e-4 | 5e-5, 2e-4 | `opt.learning_rate` in YAML |
| 2 | `n_mha_win_size` | [19]*6 | [9]*6, [29]*6 | `model.n_mha_win_size` |
| 3 | `epochs` | 30-35 | 40, 50 | `opt.epochs` |
| 4 | `batch_size` | 2 | 4 (if VRAM allows) | `loader.batch_size` |
| 5 | `d_state` (Mamba) | 16 | 8, 32 | Add to `model` section |
| 6 | `embd_dim` | 512 | 256, 384 | `model.embd_dim` |

**Target scores (from the paper):**
- ActionFormer baseline: 67.9% avg mAP
- Original Mamba: 68.5%
- Your BiMamba+Transformer: aim for 66-69%

**If stuck below 55%**, the problem is almost certainly data paths, not architecture. Run the original `thumos_i3d.yaml` config with ActionFormer's Transformer backbone first to verify your pipeline.